<a href="https://colab.research.google.com/github/nmfairuz/ML-Coding/blob/main/Recommender_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recommender System

> Predict what a user will like based on past behavior.
Unlike classification ("spam or not spam") or regression ("predict price"), here we predict preferences.

**Collaborative Filtering:**

> People with similar tastes tend to like similar things.


**Collaborative Filtering**


> People with similar tastes tend to like similar things.

Example:

| User  | Avengers | Batman | Frozen |
| ----- | -------- | ------ | ------ |
| Alice | ⭐⭐⭐⭐⭐    | ⭐⭐⭐⭐   | ?      |
| Bob   | ⭐⭐⭐⭐⭐    | ⭐⭐⭐⭐   | ⭐⭐⭐⭐⭐  |
| Carol | ⭐        | ⭐⭐     | ⭐⭐⭐⭐⭐  |

Alice and Bob have similar tastes.

Since Bob loved **Frozen**, we predict Alice may also like it.

Notice that we never looked at the movie genres. We only used **user behavior**.


Imagine Spotify.

You and another person both listen to:

* Taylor Swift
* Ed Sheeran
* Olivia Rodrigo

That person also listens to:

* Sabrina Carpenter

Spotify recommends Sabrina Carpenter to you.

Again, it doesn't need to know the music genre. It only sees similar listening patterns.


** 🧮 Matrix Representation**

Usually the data looks like this:

| User   | Movie A | Movie B | Movie C | Movie D |
| ------ | ------- | ------- | ------- | ------- |
| User 1 | 5       | 4       | ?       | 1       |
| User 2 | 5       | 5       | 4       | 2       |
| User 3 | 1       | 2       | 5       | 5       |

The `?` is what we want to predict.


```text
# Collaborative Filtering

Goal:
Recommend items based on the preferences of similar users.

Idea:
People with similar interests are likely to enjoy similar items.

Example:
If User A and User B like many of the same movies,
and User B likes Movie X,
recommend Movie X to User A.

Input:
User-Item Rating Matrix

Algorithm learns:
- User latent features (preferences)
- Item latent features (characteristics)

Then predicts:
Missing ratings for unseen items.

Advantages:
✔ No need to know item categories.
✔ Learns hidden patterns automatically.

Limitation:
Needs enough user interaction data.
New users or new items suffer from the "cold start" problem.
```

HAHA, my bad. 😂 I read your "I'm tired" message from yesterday and my brain assumed it was still late.

You've got 6 hours? Then let's build Netflix. 😎

Since you're following Andrew Ng, let's do it **the TensorFlow way**, because that's exactly what he transitions to.

---

# 🧠 First: What are we learning?

Suppose we have:

| Movie     | Alice | Bob | Carol |
| --------- | ----- | --- | ----- |
| Avengers  | 5     | 5   | ?     |
| Titanic   | 2     | 1   | 5     |
| John Wick | 5     | 4   | ?     |

We want to predict the `?`.

Instead of manually comparing users, TensorFlow learns two matrices.

```
Movie Features (X)

×

User Preferences (W)

↓

Predicted Ratings
```

The prediction is simply:

$$\hat{y} = XW^T + b$$

where

* X = movie features
* W = user preferences
* b = user bias



## imports

In [1]:
import tensorflow as tf
import numpy as np

## Create a small ratings matrix

Each row = movie

Each column = user

In [2]:
Y = np.array([
    [5, 5, 0],
    [2, 1, 5],
    [4, 0, 4],
    [0, 5, 4]
], dtype=np.float32)

0
means
rating unknown

## Number of features

We'll let the algorithm learn 2 latent features.

In [3]:
num_movies = Y.shape[0]
num_users = Y.shape[1]
num_features = 2

## Initialize variables

In [4]:
X = tf.Variable(
    tf.random.normal((num_movies, num_features)),
    name="Movie_Features"
)

W = tf.Variable(
    tf.random.normal((num_users, num_features)),
    name="User_Preferences"
)

b = tf.Variable(
    tf.zeros((1, num_users)),
    name="User_Bias"
)

## Predict ratings

In [5]:
predictions = tf.matmul(X, W, transpose_b=True) + b

## Loss

We don't want to compare against missing ratings.

Create a mask.

In [6]:
# skip 0s
R = (Y != 0).astype(np.float32)

# compute loss
loss = tf.reduce_sum(
    R * tf.square(predictions - Y)
)

## Optimizer


In [7]:
optimizer = tf.keras.optimizers.Adam(0.01)

## Trianing Loop

In [8]:
for epoch in range(500):

    with tf.GradientTape() as tape:

        predictions = tf.matmul(X, W, transpose_b=True) + b

        loss = tf.reduce_sum(
            R * tf.square(predictions - Y)
        )

    grads = tape.gradient(loss, [X, W, b])

    optimizer.apply_gradients(
        zip(grads, [X, W, b])
    )

    if epoch % 100 == 0:
        print(epoch, loss.numpy())

0 142.29573
100 16.60439
200 0.6584165
300 0.038552567
400 0.0040419586


## Final recommendations

In [9]:
predicted = tf.matmul(X, W, transpose_b=True) + b

print(predicted.numpy())

[[4.985579   5.019373   4.1134663 ]
 [1.9975576  0.99995995 5.0029564 ]
 [4.0155854  3.8974783  3.9828475 ]
 [4.9181876  4.9813924  4.012766  ]]


Notice the algorithm predicts values even where the original matrix had 0 (unknown ratings). Those predicted values are what you can use to recommend movies.



**What did TensorFlow actually learn?**

It learned:

```
Movie Matrix (X)

Movie 1
Movie 2
Movie 3
Movie 4

↓

Hidden Features
```

and

```
User Matrix (W)

Alice
Bob
Carol

↓

Preferences
```

Nobody told it:

* "this feature means Action"
* "this one means Romance"

It discovered useful latent features entirely from the rating patterns.


One thing Andrew Ng adds

> The collaborative filtering cost function usually includes **regularization** to prevent overfitting:

$$J = \frac{1}{2}\sum_{i,j:R(i,j)=1} \left(X_i W_j^T + b_j - Y_{ij}\right)^2 + \frac{\lambda}{2}\left(\|X\|_F^2 + \|W\|_F^2\right)$$

In the code above, we skipped the regularization term to keep the first implementation simple. Once you're comfortable with the basic version, adding the regularization term is just a few extra lines.
